# B4: Grassland Harvest (monthly)

Estimate monthly grazing HANPPharv over grassland.
- Source: FAO livestock statistics + grassland cover
- Method: distribute grazing pressure by grassland NPP seasonality
- Output: monthly grassland HANPPharv (g C m⁻²) → `interim/`

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

# GEE MODIS data

In [ ]:
da_luc = (
    xr.open_dataarray(cfg["paths"]["gee_exports"] / f"luc/luc-500m.tif")
        .chunk({"x": 1000, "y": 1000})
        .assign_coords(band=range(2001, 2025))
        .rename({"band": "year"})
        .drop_attrs()
        .rio.write_crs("EPSG:4326")
)

# Standard map

In [ ]:
gdf_world_merged = (
    gpd.read_parquet(cfg["paths"]["output"] / "world_clean.parquet")
        .query("NE_name != 'Antarctica'")
        .reset_index(drop=True)
        .assign(idx=lambda x: np.arange(1, x.shape[0]+1))    
)

In [ ]:
da_template = create_target_grid()
da_countries = make_geocube(gdf_world_merged, measurements=["idx"], like=da_template)["idx"].fillna(0)

# Grassland area fraction

In [ ]:
"""Allocate F4 national monthly grassland carbon harvest to the (month, y, x) grid."""

# MODIS LC_Type1 grassland classes (IGBP)
# 6   #c6b044   Closed Shrublands: dominated by woody perennials (1-2m height) >60% cover.
# 7   #dcd159   Open Shrublands: dominated by woody perennials (1-2m height) 10-60% cover.
# 8   #dade48   Woody Savannas: tree cover 30-60% (canopy >2m).
# 9   #fbff13   Savannas: tree cover 10-30% (canopy >2m).
# 10  #b6ff05   Grasslands: dominated by herbaceous annuals (<2m).
# 14  #ff6d4c   Cropland/Natural Vegetation Mosaics: mosaics of small-scale cultivation 40-60% with natural tree, shrub, or herbaceous vegetation.
GRASS_CLASSES = [7, 8, 9, 10, 14]   
    
def _load_grass_frac(year):
    """
    MCD12Q1 LC_Type1 (0.05deg) -> grassland binary mask -> coarsen(22x22, mean) -> 0.1deg grassland area fraction.
    mean not sum: after coarsen the range stays [0,1] = grassland fraction in the cell.
    """
    lc = (
        xr.open_dataarray(cfg["paths"]["gee_exports"] / f"luc/luc-500m.tif")
        .chunk({"x": 1000, "y": 1000})
        .assign_coords(band=range(2001, 2025))
        .rename({"band": "year"})
        .sel(year=year)
        .drop_vars("year")
        .drop_attrs()
        .rio.write_crs("EPSG:4326")
    )
    
    da_WDPA = xr.open_dataarray(
        cfg["paths"]["output"] / f"protected_areas/WDPA_{year}.tif", 
        chunks={"x": 1000, "y": 1000}).fillna(0)
    
    da_grass_frac = (
        (lc.isin(GRASS_CLASSES) * da_WDPA)
        .astype(np.float32)
        .coarsen(x=22, y=22, boundary="trim")
        .mean()                          # 500m x22 ~ 0.099deg; mean = grassland area fraction
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .fillna(0)
    )
    return da_grass_frac

In [ ]:
da_grass = (
    xr.open_dataarray(cfg["paths"]["gee_exports"] / f"luc/luc-500m.tif")
    .chunk({"x": 1000, "y": 1000})
    .assign_coords(band=range(2001, 2025))
    .rename({"band": "year"})
    .sel(year=slice(2010, 2024))
    .drop_attrs()
    .rio.write_crs("EPSG:4326")
)

In [ ]:
da_WDPA_lst = []
for year in range(2010, 2025):
    da_WDPA = xr.open_dataarray(cfg["paths"]["output"] / f"protected_areas/WDPA_{year}.tif", chunks={"x": 1000, "y": 1000})\
        .sel(band=1).drop_vars("band").fillna(0).astype(np.uint8)\
        .drop_attrs()
    da_WDPA_lst.append(da_WDPA)
da_WDPA = xr.concat(da_WDPA_lst, dim="year").assign_coords(year=range(2010, 2025))

In [ ]:
da_grass_frac = (
    (da_grass.isin(GRASS_CLASSES) * da_WDPA)
    .astype(np.float32)
    .coarsen(x=22, y=22, boundary="trim")
    .mean()                          # 500m x22 ~ 0.099deg; mean = grassland area fraction
    .rio.reproject_match(da_template, resampling=Resampling.nearest)
    .fillna(0)
)

In [ ]:
da_grass_frac.to_dataset(name="grass_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/grass_frac.nc", 
        encoding={"grass_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

# Total grassland harvest

In [ ]:
def get_fao_annual(year):
    df_grass_harv_C = (
        pd.read_csv(cfg["paths"]["output"] / "total/graze.csv")
        .set_index("Area")
        .join(gdf_world_merged.set_index("FAO_name")["idx"])
        .drop(columns=["PNAS_region"])
        .sort_values("idx")
        .reset_index()
        [["Area", "idx", f"{year}"]]
    )

    fao_annual = df_grass_harv_C.set_index("idx")[f"{year}"].to_dict()
    return fao_annual

year = 2015
fao_annual = get_fao_annual(year)

# Livestock distribution weight

In [ ]:
dic_livestock = {'Asses': 'Asses', 'Camels': 'Camels', 'Cattle': 'Cattle and Buffaloes', 'Chickens': 'Poultry Birds', 'Goats': 'Sheep and Goats', 'Horses': 'Horses', 'Mules and hinnies': 'Mules and hinnies', 'Sheep': 'Sheep and Goats', 'Bees': 'Other rodents', 'Buffalo': 'Cattle and Buffaloes', 'Ducks': 'Poultry Birds', 'Geese': 'Poultry Birds', 'Swine / pigs': 'Swine / pigs', 'Turkeys': 'Poultry Birds', 'Rabbits and hares': 'Rabbits and hares', 'Other camelids': 'Other camelids', 'Other rodents': 'Other rodents', 'Other birds': 'Poultry Birds'}
df_livestock_factor = pd.read_excel(Path("../data/params/grazing_coefficients.xlsx"), sheet_name="livestock")
df_livestock_factor = df_livestock_factor.melt(id_vars="Unnamed: 0")\
    .rename(columns={"Unnamed: 0": "Live_cls", "variable": "PNAS_region", "value": "fi"})\
    .assign(fi=lambda _df: _df["fi"]*1000)


In [ ]:
def _make_fi_raster(live_cls):
    fi_map = (
        df_livestock_factor
        .query("Live_cls == @live_cls")
        .set_index("PNAS_region")["fi"]
    )
    gdf = gdf_world_merged.assign(fi=lambda x: x["PNAS_region"].map(fi_map))
    return make_geocube(gdf, measurements=["fi"], like=da_template)["fi"]

da_fi_cattle = _make_fi_raster("Cattle and Buffaloes")
da_fi_sheep  = _make_fi_raster("Sheep and Goats")
da_fi_horse  = _make_fi_raster("Horses")


In [ ]:
dic_livestock_map2feed_factor = {
    'Buffa': 'Cattle and Buffaloes',
    'Cattl': 'Cattle and Buffaloes',
    'Chick': 'Poultry Birds',
    'Ducks': 'Poultry Birds',
    'Goats': 'Sheep and Goats',
    'Horse': 'Horses',
    'Sheep': 'Sheep and Goats',
    'Swine': 'Swine / pigs'
}
sel_livestock = ['Cattle and Buffaloes', 'Sheep and Goats', 'Horses']
df_livestockmap_file = (
    pd.DataFrame([(i.stem, i) for i in cfg["paths"]["livestockmap"].glob("*.tif")], columns=["fname", "fpath"])
    .assign(
        livestock_type=lambda x: x["fname"].str.split("_").str[0], 
        year=lambda x: x["fname"].str.split("_").str[1].astype(int),
        Live_cls=lambda x: x["livestock_type"].map(dic_livestock_map2feed_factor))
    .query("year >= 2010")
    .query("Live_cls in @sel_livestock")
)


In [ ]:
year = 2015
df_livestockmap_file.query("year == @year")

In [ ]:
def cal_livestock_w_1year(year):
    def _open_livestock(year, lt):
        fp = df_livestockmap_file.query("year == @year and livestock_type == @lt")["fpath"].iloc[0]
        return xr.open_dataarray(fp, chunks="auto").sel(band=1).drop_vars("band").fillna(0)


    da_cattle = (
        sum(_open_livestock(year, lt) for lt in ["Buffa", "Cattl"])
        .coarsen(x=2, y=2, boundary="trim").sum()
        .rio.write_crs("EPSG:4326")
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .fillna(0)
    )

    da_sheep = (
        sum(_open_livestock(year, lt) for lt in ["Sheep", "Goats"])
        .coarsen(x=2, y=2, boundary="trim").sum()
        .rio.write_crs("EPSG:4326")
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .fillna(0)
    )

    da_horse = (
        sum(_open_livestock(year, lt) for lt in ["Horse"])
        .coarsen(x=2, y=2, boundary="trim").sum()
        .rio.write_crs("EPSG:4326")
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .fillna(0)
    )
    da_livestock_weight = da_cattle * da_fi_cattle.values + da_sheep * da_fi_sheep.values + da_horse * da_fi_horse.values
    return da_livestock_weight
    
da_livestock_weight=cal_livestock_w_1year(2020)
da_livestock_weight.plot.imshow()


In [ ]:
lst_da_livestock_weight = []
for year in range(2010, 2022):
    da_livestock_weight = cal_livestock_w_1year(year)
    lst_da_livestock_weight.append(da_livestock_weight)

In [ ]:
lst_da_livestock_weight_2021 = lst_da_livestock_weight[-1]
lst_da_livestock_weight_2010_2024 = [*lst_da_livestock_weight, lst_da_livestock_weight_2021, lst_da_livestock_weight_2021, lst_da_livestock_weight_2021]

In [ ]:
da_livestock_weight_merge = xr.concat(lst_da_livestock_weight_2010_2024, dim="year").assign_coords(year=range(2010, 2025))

In [ ]:
da_livestock_weight_merge.to_dataset(name="livestock_weight")\
    .to_netcdf(
        cfg["paths"]["output"] / f"livestock_weight.nc", 
        encoding={"livestock_weight": {"dtype": "float32", "zlib": True, "complevel": 4}})

# GRASSLAND HARVEST CALCULATION 

In [ ]:
year = 2010
grass_frac = (
    xr.open_dataset(cfg["paths"]["output"] / f"WDPA_masked/grass_frac.nc")["grass_frac"]
    .sel(year=year)
    .drop_vars("year")
    .drop_attrs()
    .rio.write_crs("EPSG:4326")
)

In [ ]:
"""
GrasslandAllocator
Allocate national annual grassland carbon harvest to monthly 0.1-degree grid cells.

overall workflow:
    1. load PSN_monthly / MODIS land cover, align to 0.1deg
    2. build spatial weights:
           W[mo, y, x] = grass_frac[y, x] × PSN[mo, y, x]
       meaning: how much of the national grassland-harvest quota a cell receives in month mo,
             determined jointly by grassland-area fraction x monthly forage potential
    3. vectorized allocation (bincount, no per-country loop):
           result[mo, y, x] = FAO_annual[c] × W[mo, y, x] / Σ_W[c, all months]

data sources:
    PSN       : GEE-exported MODIS MOD17A2HGF monthly net photosynthesis (gC/m2)
    MODIS LC  : MCD12Q1 LC_Type1, 0.05deg -> coarsen to 0.1deg
    FAO output: national annual grassland carbon harvest totals

usage:
    ga = GrasslandAllocator(cfg, year=2015)
    ga.load()
    result = ga.allocate(fao_annual, country_mask)  # → DataArray (12, ny, nx)
"""

class GrasslandAllocator:
    """Allocate national annual grassland carbon totals over month, y, and x."""

    # MODIS LC_Type1 grassland classes (IGBP)
    # 6   #c6b044   Closed Shrublands: dominated by woody perennials (1-2m height) >60% cover.
    # 7   #dcd159   Open Shrublands: dominated by woody perennials (1-2m height) 10-60% cover.
    # 8   #dade48   Woody Savannas: tree cover 30-60% (canopy >2m).
    # 9   #fbff13   Savannas: tree cover 10-30% (canopy >2m).
    # 10  #b6ff05   Grasslands: dominated by herbaceous annuals (<2m).
    # 14  #ff6d4c   Cropland/Natural Vegetation Mosaics: mosaics of small-scale cultivation 40-60% with natural tree, shrub, or herbaceous vegetation.
    GRASS_CLASSES = [7, 8, 9, 10, 14]   

    def __init__(self, cfg: dict, year: int):
        self.year = year
        self.da_template = create_target_grid()
        self.grid_area   = self.da_template.values   # (ny, nx), unit m2
        self.target_y    = self.da_template.y.values
        self.target_x    = self.da_template.x.values

        self.psn_path = (Path(cfg["paths"]["gee_exports"])
                         / f"PSN_monthly_{year}.tif")
        self.lc_path  = Path(cfg["paths"]["gee_exports"]) / "luc/luc-500m.tif"

        self.psn        = None   # DataArray (12, ny, nx)
        self.grass_frac = None   # DataArray     (ny, nx)

    # ==================================================================
    # 1. load data
    # ==================================================================

    DAYS_IN_MONTH = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])

    def load(self):
        self._load_psn()
        self._load_grass_frac()
        self._load_liestock_weight()
        
    def _load_psn(self):
        """same as B4: daily mean -> monthly total, clip negatives to 0."""
        days = xr.DataArray(
            self.DAYS_IN_MONTH, dims=["month"],
            coords={"month": np.arange(1, 13)},
        )
        raw = xr.open_dataarray(self.psn_path)
        self.psn = (
            raw
            .drop_attrs()
            .rio.write_crs("EPSG:4326")
            .assign_coords(band=np.arange(1, 13))
            .rename({"band": "month"})
            .clip(min=0)
            .rio.reproject_match(self.da_template, resampling=Resampling.nearest)
            * days
        )
    def _load_liestock_weight(self):
        self.livestock_weight = (
            xr.open_dataset(cfg["paths"]["output"] / f"livestock_weight.nc")["livestock_weight"]
            .sel(year=self.year)
            .drop_vars("year")
            .drop_attrs()
            .rio.write_crs("EPSG:4326")
            .rio.reproject_match(self.da_template, resampling=Resampling.nearest)
        )
        
    def _load_grass_frac(self):
        """
        MCD12Q1 LC_Type1 (0.05deg) -> grassland binary mask -> coarsen(22x22, mean) -> 0.1deg grassland area fraction.
        mean not sum: after coarsen the range stays [0,1] = grassland fraction in the cell.
        """
        self.grass_frac = (
            xr.open_dataset(cfg["paths"]["output"] / f"WDPA_masked/grass_frac.nc")["grass_frac"]
            .sel(year=self.year)
            .drop_vars("year")
            .drop_attrs()
            .rio.write_crs("EPSG:4326")
        )
        
    # ==================================================================
    # 2. build spatial weights
    # ==================================================================

    def build_spatial_weight(self) -> np.ndarray:
        """
        W[mo, y, x] = grass_frac[y, x] × grid_area[y, x] × PSN[mo, y, x]

        grass_frac x grid_area = actual cell grassland area (m2)
        times PSN = grassland-area-weighted monthly forage potential
        identical logic to B4 using harvarea (ha) as spatial weight.
        """
        actual_grass_area = (
            self.grass_frac.values          # (ny, nx)
            * self.grid_area                # (ny, nx)
        )                                   # unit: m2

        return (
            actual_grass_area[np.newaxis, :, :]               # (1,  ny, nx)
            * self.psn.values                                 # (12, ny, nx)
            * self.livestock_weight.values[np.newaxis, :, :]  # (1,  ny, nx)
        ).astype(np.float64)

    # ==================================================================
    # 3. vectorized allocation
    # ==================================================================

    def allocate(self,
                 fao_annual: dict,
                 country_mask: xr.DataArray) -> xr.DataArray:
        """
        core formula:
            result[mo, y, x] = FAO_annual[c] × W[mo, y, x] / Σ_W[c, all months]

        Parameters
        ----------
        fao_annual   : {country_idx: annual carbon harvest in gC}
        country_mask : (ny, nx) DataArray, values are country idx (integer)

        Returns
        -------
        DataArray (12, ny, nx)  per-cell per-month grassland carbon harvest (gC)
        """
        W    = self.build_spatial_weight()                        # (12, ny, nx)
        mask = country_mask.fillna(0).values.astype(np.intp)     # (ny, nx)
        ny, nx  = mask.shape
        n_ids   = int(mask.max()) + 1
        flat    = mask.ravel()

        # -- 1. build the annual FAO lookup array by country id --
        fao_arr = np.zeros(n_ids, dtype=np.float64)
        for code, annual_val in fao_annual.items():
            code_int = int(code)
            if 0 <= code_int < n_ids:
                fao_arr[code_int] = annual_val  # scalar
                
        # -- 2. annual total weight summed over all months --
        country_total = np.zeros(n_ids, dtype=np.float64)
        for mo in range(12):
            country_total += np.bincount(flat, weights=W[mo].ravel(), minlength=n_ids)
        
        # -- 3. vectorized: result = FAO x W[mo,y,x] / sum_W[c,annual] --
        safe_total = np.where(country_total > 0, country_total, np.inf)
        ratio = (fao_arr / safe_total)[flat].reshape(ny, nx)  # (ny, nx)
        result = W * ratio[np.newaxis, :, :]                   # (12, ny, nx)

        return xr.DataArray(
            result,
            dims=["month", "y", "x"],
            coords={"month": np.arange(1, 13),
                    "y": self.target_y,
                    "x": self.target_x},
            name="grassland_HANPPharv",
        )

# Test

In [ ]:
da_template = create_target_grid()
da_countries = make_geocube(gdf_world_merged, measurements=["idx"], like=da_template)["idx"].fillna(0)

In [ ]:
year = 2015
fao_annual = get_fao_annual(year)
ga = GrasslandAllocator(cfg, year=year)
ga.load()
da_grass_harvest = ga.allocate(fao_annual, da_countries) / da_template

In [ ]:
dic_idx2FAO_name = gdf_world_merged.set_index("idx")["FAO_name"].to_dict()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
da_grass_harvest_fl = da_grass_harvest.values.flatten()
vmax_val = np.nanpercentile(da_grass_harvest_fl[da_grass_harvest_fl > 0], 99) if (da_grass_harvest_fl > 0).any() else 1
vmax_val = 100
for mo in range(12):
    ax = axes.flat[mo]
    data = np.where(da_grass_harvest[mo] > 0, da_grass_harvest[mo], np.nan)
    im = ax.imshow(data,
              extent=(da_template.x.min(), da_template.x.max(),
                      da_template.y.min(), da_template.y.max()),
              cmap="YlOrRd", vmin=0, vmax=vmax_val)
    ax.invert_yaxis()
    ax.set_title(f"month {mo+1}", fontsize=10)
    ax.tick_params(labelsize=7)
plt.tight_layout()
plt.colorbar(im)

# Main pipeline

In [ ]:
def cal_harv_1year(year):
    fao_annual = get_fao_annual(year)
    ga = GrasslandAllocator(cfg, year=year)
    ga.load()
    da_grass_harvest = ga.allocate(fao_annual, da_countries) / da_template
    ds_HANPPharv = xr.Dataset({"graze": da_grass_harvest})

    # save
    encoding = {
        var: {"dtype": "float32", "zlib": True, "complevel": 4, "chunksizes": (1, 180, 360)}
        for var in ds_HANPPharv.data_vars
    }
    ds_HANPPharv.to_netcdf(
        Path(cfg["paths"]["output"]) / f"HANPPharv_graze_{year}.nc",
        encoding=encoding
    )

In [ ]:
for year in range(2010, 2025):
    _nc_path = Path(cfg["paths"]["output"]) / f"HANPPharv_graze_{year}.nc"
    if _nc_path.exists():
        continue
    print(f"\n=== Calculating harvest for year {year} ===")
    cal_harv_1year(year)